## Set Up Environment

In [ ]:
# imports
from sklearn.neural_network import MLPClassifier
from pathlib import Path
import pandas as pd
import numpy as np
import sys

# custom imports
DATA_LOADING_DIRECTORY = (Path("./Preprocessing/").resolve())
sys.path.append(str(DATA_LOADING_DIRECTORY))
from data_loading import *
from encode import *
from Evaluation.cross_validation import get_outer_train_test_split, cross_val_test, auc_score

# features
from sklearn.preprocessing import QuantileTransformer
from FeatureEngineering.diagnosis_graph import DiagnosisGraph
from FeatureEngineering.clinical_features import full_clinical_feature_engineering
from Preprocessing.scale import standardise_continuous_features


In [ ]:
# config for cross validation
FOLD_COLUMN = "fold"
NUM_FOLDS = 10

# target column
TARGET_COLUMN = "readmitted_30_days"

In [ ]:
# read data 
FILENAME = 'diabetes_stratified' 
FOLDER = 'Stratified'
df_fold = read_data(FILENAME, FOLDER)

## MLP Wrapper Class

#### Class Config

In [ ]:
# hyper parameter configuration
default_params = {
    # main tuning 
    "hidden_layer_sizes": (32,),
    "balance":None,

    # other fixed hyper parameters for lightweight MLP
    "alpha": 1e-2,
    "batch_size": 512,
    "learning_rate_init": 1e-3,
    "activation": "relu",
    "solver": "adam",
    "early_stopping": True,
    "validation_fraction": 0.1,
    "n_iter_no_change": 5,
    "max_iter": 100,
    "random_state": 42,
}

# target column
TARGET_COLUMN = "readmitted_30_days"

#### Class Code

In [ ]:
'''
WRAPPER CLASS FOR MLP WITH DATASET CLASS RESAMPLING 
'''
class MLPClassifierWrapper:
    def __init__(self, params={}):    
        '''
        Initialise MLP with chosen params otherwise use defaults
        '''    
        params = {**default_params, **params}
        self.params = params
        
        # write
        self.balance = params["balance"]
        self.model = MLPClassifier(
            # main hyper params
            hidden_layer_sizes=params["hidden_layer_sizes"],
            alpha=params["alpha"],

            # extra hyperparameters to fix for a lightweight MLP
            activation=params["activation"],
            batch_size=params["batch_size"],
            learning_rate_init=params["learning_rate_init"],
            early_stopping=params["early_stopping"],
            max_iter=params["max_iter"],
            random_state=params["random_state"],
            solver=params["solver"],
            validation_fraction=params["validation_fraction"],
            n_iter_no_change=params["n_iter_no_change"],
        )

    def fit(self, X_train, y_train, MINORITY_CLASS=1):
        '''
        fit MLP, but optionally resample training data to ensure
        fixed frequency of target class (to fix imbalance)
        '''
        if self.balance != None:
            # avoid mutating original data
            X_train_fit = X_train.copy()
            X_train_fit[TARGET_COLUMN] = y_train.values

            # calculate required samples
            X_sample = X_train_fit.loc[X_train_fit[TARGET_COLUMN]==MINORITY_CLASS]
            num_majority_class = len(X_train)-len(X_sample)
            num_minority_class = len(X_sample)
            num_oversample = int(self.balance/(1-self.balance) * num_majority_class - num_minority_class) # could integrate balance
            
            # add target samples to training set
            if num_oversample > 0:
                X_oversample = X_sample.sample(n=num_oversample, random_state=42, replace=True)
                X_train_fit = pd.concat([X_train_fit, X_oversample])

            # finalise data and fit model
            y_train_fit = X_train_fit[TARGET_COLUMN]
            X_train_fit.drop(columns=[TARGET_COLUMN], inplace=True)
            self.model.fit(X_train_fit, y_train_fit)
        else:
            # fit as normal
            self.model.fit(X_train, y_train)
            
    def predict(self, X_test):
        '''
        predict wrapper, estimates probability of 30-day readmission
        '''
        return self.model.predict_proba(X_test)[:,1]    
    
def MLP_keep_columns(df_train, df_test):
    '''
    only keep columns with data types compatible with MLP
    '''
    return (df_train.select_dtypes(include=["number", "bool"]), 
            df_test.select_dtypes(include=["number", "bool"]))

def standardise_features_for_mlp(df_train, df_test):
    '''
    Standardise numeric model features while preserving target and fold columns.
    '''
    target_col = TARGET_COLUMN
    fold_col = FOLD_COLUMN

    # store labels separately so they are not scaled
    train_target = df_train[target_col].reset_index(drop=True)
    train_fold = df_train[fold_col].reset_index(drop=True)

    test_target = df_test[target_col].reset_index(drop=True)
    test_fold = df_test[fold_col].reset_index(drop=True)

    # isolate model inputs before selecting numeric features
    X_train = df_train.drop(columns=[target_col, fold_col]).reset_index(drop=True)
    X_test = df_test.drop(columns=[target_col, fold_col]).reset_index(drop=True)

    numeric_cols = X_train.select_dtypes(include="number").columns

    X_train = X_train[numeric_cols].copy()
    X_test = X_test[numeric_cols].copy()

    # fit scaling on train only, then apply same transform to test
    X_train, X_test, _ = standardise_continuous_features(
        X_train,
        X_test,
    )

    # reattach target and fold columns for downstream CV logic
    df_temp1 = pd.DataFrame({target_col: train_target.values, fold_col: train_fold.values})
    df_train_scaled = pd.concat(
        [X_train.reset_index(drop=True), df_temp1], axis=1,
    )

    df_temp2 = pd.DataFrame({target_col: test_target.values, fold_col: test_fold.values})
    df_test_scaled = pd.concat(
        [X_test.reset_index(drop=True), df_temp2], axis=1
    )
    return df_train_scaled, df_test_scaled

## Cross Val Tuning and Test

#### Hyper parameter grid

In [ ]:
# config for hyperparameter tuning
HIDDEN_LAYER_GRID = [
    (4,), (8,), (16,)
]
BALANCE_GRID = [
    None, 0.35, 0.45, 0.55
]

# build grid
params_grid = []
for hidden_layer in HIDDEN_LAYER_GRID:
    for balance in BALANCE_GRID:
        new_params = {}
        new_params['hidden_layer_sizes'] = hidden_layer
        new_params['balance'] = balance
        params_grid.append(new_params)

In [ ]:
# config for cross val processing
data_processing = [full_reduce_df, full_ordinal_encode, full_one_hot_encode, standardise_features_for_mlp, MLP_keep_columns]

#### Run for Results on Baseline Data

In [ ]:
'''
results_df_baseline, predictions_df_baseline = cross_val_test(
    ModelClass=MLPClassifierWrapper,
    df_fold=df_fold,
    params_grid=params_grid,
    data_processing=data_processing,
    num_folds=10,
    target_column=TARGET_COLUMN,
    fold_column=FOLD_COLUMN,
    eval_metric=auc_score,
    eval_uses_scores=True,
    inner_splits=3,
    tune_threshold=True,
    threshold_metric="balanced_accuracy",
    default_threshold=0.5,
    annotations=True,
)
#'''

In [ ]:
#save_data(results_df_baseline, "MLP_baseline_results_v2", "Results")

#### Results with Graph Features

In [ ]:
# separate for the moment
def preprocess_fold(df_train, df_valid, RANDOM_STATE=42):
    '''
    Full preprocessing pipeline for a single train/validation split.
    All transformations are fitted on the training fold only.
    '''
    y_train = df_train[TARGET_COLUMN].values.astype(np.float32)
    y_valid = df_valid[TARGET_COLUMN].values.astype(np.float32)
    X_train = df_train.drop(columns=[TARGET_COLUMN, FOLD_COLUMN])
    X_valid = df_valid.drop(columns=[TARGET_COLUMN, FOLD_COLUMN])

    # remove columns with no variation
    X_train, X_valid = full_reduce_df(X_train, X_valid)

    # convert ordinal variables to numeric representations
    X_train, X_valid = full_ordinal_encode(X_train, X_valid)

    # augment diagnosis codes with graph-derived features
    graph = DiagnosisGraph()
    X_train, X_valid = graph.fit_transform_fold_augmented(
        X_train, X_valid, drop_diag_cols=True, reset_index=True,
    )

    # encode remaining categorical variables
    X_train, X_valid = full_one_hot_encode(X_train, X_valid)

    # retain numeric model inputs only
    numeric_cols = X_train.select_dtypes(include="number").columns
    X_train = X_train[numeric_cols]
    X_valid = X_valid[numeric_cols]

    # fit transformation on train and apply to validation fold
    qt = QuantileTransformer(
        output_distribution="normal",
        random_state=RANDOM_STATE,
    )
    X_train_np = qt.fit_transform(X_train.values).astype(np.float32)
    X_valid_np = qt.transform(X_valid.values).astype(np.float32)

    return X_train_np, y_train, X_valid_np, y_valid


def full_preprocess_fold(df_train, df_test):
    '''
    Apply fold preprocessing and reconstruct processed dataframes
    for compatibility with the cross-validation pipeline.
    '''
    X_train_np, y_train, X_test_np, y_test = preprocess_fold(
        df_train=df_train,
        df_valid=df_test,
    )

    X_train = pd.DataFrame(X_train_np)
    X_test = pd.DataFrame(X_test_np)

    # reattach target and fold information
    df_train_processed = X_train.copy()
    df_train_processed[TARGET_COLUMN] = y_train
    df_train_processed[FOLD_COLUMN] = df_train[FOLD_COLUMN].values
    df_test_processed = X_test.copy()
    df_test_processed[TARGET_COLUMN] = y_test
    df_test_processed[FOLD_COLUMN] = df_test[FOLD_COLUMN].values

    return df_train_processed, df_test_processed

#### New Config

In [ ]:
graph_processing = [full_preprocess_fold]

In [ ]:
'''
results_df_graph, predictions_df_graph = cross_val_test(
    ModelClass=MLPClassifierWrapper,
    df_fold=df_fold,
    params_grid=params_grid,
    data_processing=graph_processing,
    num_folds=10,
    target_column=TARGET_COLUMN,
    fold_column=FOLD_COLUMN,
    eval_metric=auc_score,
    eval_uses_scores=True,
    inner_splits=3,
    tune_threshold=True,
    threshold_metric="balanced_accuracy",
    default_threshold=0.5,
    annotations=True,
)
#'''

In [ ]:
#save_data(results_df_graph, "MLP_graph_results_v2", "Results")

## Clinical

In [ ]:
def clinical_processing(df_train, df_test):
    '''
    Apply clinical feature engineering for a train/test split.
    All fitted transformations are based on the training data only.
    '''
    target_col = TARGET_COLUMN
    fold_col = FOLD_COLUMN

    # keep metadata separate so it is not transformed
    train_target = df_train[target_col].reset_index(drop=True)
    train_fold = df_train[fold_col].reset_index(drop=True)
    test_target = df_test[target_col].reset_index(drop=True)
    test_fold = df_test[fold_col].reset_index(drop=True)

    # isolate model inputs
    X_train = df_train.drop(columns=[target_col, fold_col]).reset_index(drop=True).copy()
    X_test = df_test.drop(columns=[target_col, fold_col]).reset_index(drop=True).copy()

    # apply clinical preprocessing steps
    X_train, X_test = full_reduce_df(X_train, X_test)
    X_train, X_test = full_ordinal_encode(X_train, X_test)
    X_train, X_test = full_clinical_feature_engineering(X_train, X_test)
    X_train, X_test = full_one_hot_encode(X_train, X_test, threshold=0.01)

    # keep only numeric model inputs
    numeric_cols = X_train.select_dtypes(include="number").columns
    X_train = X_train[numeric_cols].reset_index(drop=True).copy()
    X_test = X_test[numeric_cols].reset_index(drop=True).copy()

    # standardise using training-set statistics only
    X_train, X_test, _ = standardise_continuous_features(X_train, X_test)

    # reattach target and fold information
    train_metadata = pd.DataFrame({target_col: train_target.values, fold_col: train_fold.values})
    test_metadata = pd.DataFrame({target_col: test_target.values, fold_col: test_fold.values})
    X_train = pd.concat([X_train, train_metadata], axis=1)
    X_test = pd.concat([X_test, test_metadata], axis=1)
    return X_train, X_test

def graph_clinical_processing(df_train, df_test):
    '''
    Apply combined clinical and graph feature engineering.
    Graph construction is fitted on the training split only.
    '''
    target_col = TARGET_COLUMN
    fold_col = FOLD_COLUMN

    # keep metadata separate so it is not transformed
    train_target = df_train[target_col].reset_index(drop=True)
    train_fold = df_train[fold_col].reset_index(drop=True)

    test_target = df_test[target_col].reset_index(drop=True)
    test_fold = df_test[fold_col].reset_index(drop=True)

    # isolate model inputs
    X_train = df_train.drop(columns=[target_col, fold_col]).reset_index(drop=True).copy()
    X_test = df_test.drop(columns=[target_col, fold_col]).reset_index(drop=True).copy()

    # create clinical features before graph construction
    X_train, X_test = full_reduce_df(X_train, X_test)
    X_train, X_test = full_clinical_feature_engineering(X_train, X_test)
    X_train, X_test = full_ordinal_encode(X_train, X_test)

    # build diagnosis graph on train only, then transform test
    graph = DiagnosisGraph()
    X_train, X_test = graph.fit_transform_fold_augmented(
        X_train,
        X_test,
        y_train_fold=train_target.values,
        drop_diag_cols=False,
        reset_index=True,
    )

    # encode categorical variables after graph augmentation
    X_train, X_test = full_one_hot_encode(X_train, X_test, threshold=0.01)

    # keep only numeric model inputs
    numeric_cols = X_train.select_dtypes(include="number").columns
    X_train = X_train[numeric_cols].reset_index(drop=True).copy()
    X_test = X_test[numeric_cols].reset_index(drop=True).copy()

    # standardise using training-set statistics only
    X_train, X_test, _ = standardise_continuous_features(X_train, X_test)

    # reattach target and fold information
    train_metadata = pd.DataFrame({target_col: train_target.values, fold_col: train_fold.values})
    test_metadata = pd.DataFrame({target_col: test_target.values, fold_col: test_fold.values})
    X_train = pd.concat([X_train, train_metadata], axis=1)
    X_test = pd.concat([X_test, test_metadata], axis=1)
    return X_train, X_test

#### Clinical

In [ ]:
'''
results_df_clinical, predictions_df_clinical = cross_val_test(
    ModelClass=MLPClassifierWrapper,
    df_fold=df_fold,
    params_grid=params_grid,
    data_processing=[clinical_processing],
    num_folds=10,
    target_column=TARGET_COLUMN,
    fold_column=FOLD_COLUMN,
    eval_metric=auc_score,
    eval_uses_scores=True,
    inner_splits=3,
    tune_threshold=True,
    threshold_metric="balanced_accuracy",
    annotations=True,
)
#'''

In [ ]:
#save_data(results_df_clinical, "MLP_clinical_results_v3", "Results")

#### Graph + Clinical

In [ ]:
'''
results_df_graph_clinical, predictions_df_graph_clinical = cross_val_test(
    ModelClass=MLPClassifierWrapper,
    df_fold=df_fold,
    params_grid=params_grid,
    data_processing=[graph_clinical_processing],
    num_folds=10,
    target_column=TARGET_COLUMN,
    fold_column=FOLD_COLUMN,
    eval_metric=auc_score,
    eval_uses_scores=True,
    inner_splits=3,
    tune_threshold=True,
    threshold_metric="balanced_accuracy",
    annotations=True,
)
#'''

In [ ]:
#save_data(results_df_graph_clinical, "MLP_graph_clinical_results_v3", "Results")

## Evaluation

In [ ]:
# read data
df_baseline = read_data("MLP_baseline_results", "Results")
df_graph = read_data("MLP_graph_results", "Results")
df_clinical = read_data("MLP_clinical_results", "Results")
df_graph_clinical = read_data("MLP_graph_clinical_results", "Results")
dfs_dict = {"baseline":df_baseline, 
            "graph":df_graph, 
            "clinical":df_clinical, 
            "graph+clinical":df_graph_clinical}


In [ ]:
# config
SUMMARY_COLUMNS = ["auc", "balanced_accuracy", "f1", "precision", "sensitivity", "specificity", "accuracy"]

# calculate mean and standard deviation of results
rows = []
for model, df in dfs_dict.items():
    row = {"model": model}
    for col in SUMMARY_COLUMNS:
        # calculate values
        row[col] = f"{df[col].mean():.3f}+/-{df[col].std()/np.sqrt(10):.3f}"
    rows.append(row)

df_results = pd.DataFrame(rows)
df_results